# 🤖 ActividadFinal — Clasificación de imágenes con CNN
### Notebook genérico · Compatible con los 8 datasets del curso

**Optimizado para:** Google Colab (GPU T4) · Entorno local (Windows/Linux/Mac)

---

### ⚡ Antes de empezar
1. **Colab:** activa GPU → `Entorno de ejecución → Cambiar tipo → T4 GPU`
2. Edita **solo la CELDA 0** — el resto corre automáticamente
3. Sube el ZIP de tu dataset cuando la celda lo pida

### Datasets disponibles
| # | Nombre | Clases | Área |
|---|---|---|---|
| 1 | Chest X-Ray (Neumonía) | 2 | Médico |
| 2 | Brain Tumor MRI | 4 | Médico |
| 3 | PlantVillage (Enfermedades en plantas) | 38 | Agrícola |
| 4 | Frutas frescas y podridas | 6 | Agrícola |
| 5 | Mariposas y polillas | 100 | Naturaleza |
| 6 | Tipos de clima | 11 | Naturaleza |
| 7 | Grietas en concreto | 2 | Industrial |
| 8 | Clasificación de basura | 6 | Industrial |

### Lo que genera este notebook al terminar
```
proyecto/
├── data/
│   ├── distribucion.png          ← clases del dataset
│   ├── muestras_dataset.png      ← imágenes de ejemplo
│   ├── curvas.png                ← loss, accuracy y F1 por época
│   ├── confusion_counts.png      ← matriz de confusión (conteos)
│   ├── confusion_norm.png        ← matriz de confusión (recall)
│   ├── metricas_clase.png        ← precision/recall/F1 por clase
│   ├── predicciones_grid.png     ← 16 predicciones del test set
│   ├── muestras_por_clase.png    ← una predicción por clase
│   ├── historial.csv             ← métricas por época
│   └── reporte.csv               ← precision/recall/F1 por clase
└── models/
    ├── modelo_best.pth           ← mejor checkpoint
    ├── modelo_produccion.pth     ← para FastAPI / Render
    ├── modelo.onnx               ← inferencia optimizada
    └── idx2clase.json            ← mapeo índice → clase
```

---
## CELDA 0 — Configuración (edita solo aquí)

In [ ]:
# ================================================================
# ⚙️  CONFIGURACIÓN PRINCIPAL — EDITA SOLO ESTA CELDA
# ================================================================

# ─── ENTORNO ─────────────────────────────────────────────────────
# 'colab'  → Google Colab (sube el ZIP con selector, guarda en Drive)
# 'local'  → PC local (pon la ruta del ZIP en ZIP_LOCAL_PATH)
ENTORNO = 'colab'

# ─── DATASET (1–8) ───────────────────────────────────────────────
# Elige el número del dataset que descargaste:
#   1=Neumonía  2=Tumor cerebral  3=PlantVillage  4=Frutas
#   5=Mariposas 6=Clima           7=Grietas       8=Basura
DATASET_ID = 1

# ─── RUTA LOCAL (solo si ENTORNO = 'local') ──────────────────────
ZIP_LOCAL_PATH = r'C:\Users\maria\Descargas\dataset.zip'

# ─── RUTA BASE DEL PROYECTO ──────────────────────────────────────
# Colab: se crea en /content/proyecto
# Local: cambia a donde quieras guardar los resultados
BASE_PATH_LOCAL = r'C:\Users\maria\proyecto'

# ─── DRIVE (solo si ENTORNO = 'colab') ───────────────────────────
# Carpeta de Drive donde se guardarán modelos y gráficas al terminar
DRIVE_OUTPUT = 'Mi unidad/ActividadFinal/resultados'

# ─── HIPERPARÁMETROS ─────────────────────────────────────────────
# El notebook elige automáticamente el modelo y batch según tu dataset,
# pero puedes sobreescribirlos aquí:
MODEL_OVERRIDE  = None   # None = automático | ej: 'efficientnet_b2'
BATCH_OVERRIDE  = None   # None = automático | ej: 16
EPOCHS          = 30
LR              = 3e-4
PATIENCE        = 7
SEED            = 42

# ─── SPLITS (solo si el dataset NO tiene carpetas train/val) ─────
# Porcentaje del total que se usará para validación y test
VAL_SIZE  = 0.15
TEST_SIZE = 0.15

print('✅ Configuración lista')
print(f'   Entorno    : {ENTORNO}')
print(f'   Dataset    : #{DATASET_ID}')
print(f'   Epochs     : {EPOCHS}')
print(f'   Patience   : {PATIENCE}')

---
## CELDA 1 — Instalación e imports

In [ ]:
import subprocess, sys
print('Instalando dependencias...')
subprocess.run([sys.executable,'-m','pip','install','-q',
    'timm','albumentations','torchmetrics','scikit-learn','seaborn'],
    check=True)
print('✅ Dependencias listas')

In [ ]:
import os, sys, json, shutil, zipfile, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib
if ENTORNO == 'local':
    matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
warnings.filterwarnings('ignore')

import torch, torch.nn as nn, torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torchmetrics import Accuracy, F1Score
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

# ── Semilla global ────────────────────────────────────────────────
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# ── Dispositivo ───────────────────────────────────────────────────
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️  Dispositivo : {DEVICE}')
if DEVICE == 'cuda':
    print(f'   GPU         : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM        : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('⚠️  GPU no detectada — el entrenamiento será lento en CPU')

# ── Rutas base ────────────────────────────────────────────────────
BASE_PATH = Path('/content/proyecto') if ENTORNO == 'colab' else Path(BASE_PATH_LOCAL)
for d in ['data','images','models']:
    (BASE_PATH / d).mkdir(parents=True, exist_ok=True)
print(f'\n✅ Proyecto en: {BASE_PATH}')

# ── Configuración por dataset ─────────────────────────────────────
DATASET_CFG = {
    1: {'nombre':'Chest X-Ray (Neumonía)',     'modelo':'mobilenetv3_large_100','batch':32,'img':224,
        'clases':['NORMAL','PNEUMONIA'],
        'notas':'Dataset binario. Alta accuracy esperada (~92-95%).'},
    2: {'nombre':'Brain Tumor MRI',            'modelo':'efficientnet_b2',      'batch':32,'img':224,
        'clases':['glioma','meningioma','notumor','pituitary'],
        'notas':'4 clases de tumor. Imágenes en escala de grises convertidas a RGB.'},
    3: {'nombre':'PlantVillage',               'modelo':'efficientnet_b4',      'batch':32,'img':224,
        'clases':None,  # se detectan automáticamente
        'notas':'38 clases de enfermedades. Usa la carpeta color/.'},
    4: {'nombre':'Frutas frescas y podridas',  'modelo':'efficientnet_b2',      'batch':32,'img':224,
        'clases':None,
        'notas':'6 clases. Alta accuracy esperada por diferencias visuales marcadas.'},
    5: {'nombre':'Mariposas y polillas',       'modelo':'efficientnet_b2',      'batch':32,'img':224,
        'clases':None,
        'notas':'100 clases. Reto real de clasificación de especies similares.'},
    6: {'nombre':'Tipos de clima',             'modelo':'mobilenetv3_large_100','batch':32,'img':224,
        'clases':None,
        'notas':'11 clases de fenómenos meteorológicos. Sin splits previos.'},
    7: {'nombre':'Grietas en concreto',        'modelo':'mobilenetv3_large_100','batch':64,'img':224,
        'clases':['Negative','Positive'],
        'notas':'Binario. Muy alta accuracy esperada (~98%). Útil para inspección industrial.'},
    8: {'nombre':'Basura y reciclaje',         'modelo':'efficientnet_b2',      'batch':32,'img':224,
        'clases':None,
        'notas':'6 categorías de residuos. Sin splits previos.'},
}

cfg        = DATASET_CFG[DATASET_ID]
MODEL_NAME = MODEL_OVERRIDE or cfg['modelo']
BATCH_SIZE = BATCH_OVERRIDE or cfg['batch']
IMG_SIZE   = cfg['img']
NUM_WORKERS = 0 if ENTORNO == 'local' else 2

print(f'\n📦 Dataset #{DATASET_ID}: {cfg["nombre"]}')
print(f'   Modelo     : {MODEL_NAME}')
print(f'   Batch size : {BATCH_SIZE}')
print(f'   Nota       : {cfg["notas"]}')

---
## CELDA 2 — Carga y normalización automática del dataset

Esta celda:
1. Recibe tu ZIP (upload en Colab, ruta local en PC)
2. Detecta automáticamente la estructura interna
3. La convierte al formato estándar `train/val/test/clase/`
4. Verifica que todo quedó bien antes de continuar

> ⚠️ **En Colab:** aparecerá un selector de archivos — elige tu ZIP del dataset.

In [ ]:
# ================================================================
# 📥 CARGA DEL ZIP
# ================================================================

IMAGES_PATH = BASE_PATH / 'images'
ZIP_TMP     = BASE_PATH / 'tmp_zip'
ZIP_TMP.mkdir(exist_ok=True)

if ENTORNO == 'colab':
    print('📂 Selecciona tu archivo ZIP del dataset...')
    from google.colab import files
    uploaded  = files.upload()
    zip_name  = list(uploaded.keys())[0]
    zip_path  = Path(zip_name)
else:
    zip_path = Path(ZIP_LOCAL_PATH)
    if not zip_path.exists():
        raise FileNotFoundError(f'ZIP no encontrado: {zip_path}\nVerifica ZIP_LOCAL_PATH en CELDA 0')

print(f'\n📦 Descomprimiendo {zip_path.name} ({zip_path.stat().st_size/1e6:.0f} MB)...')
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(str(ZIP_TMP))
if ENTORNO == 'colab' and zip_path.exists():
    zip_path.unlink()  # liberar espacio
print('✅ Descompresión completada')

In [ ]:
# ================================================================
# 🔍 DETECTOR Y NORMALIZADOR AUTOMÁTICO DE ESTRUCTURA
# ================================================================
# Soporta todas las variantes encontradas en los 8 datasets:
#  A) Ya tiene train/ val/ test/  → usar directo
#  B) Tiene train/ pero no val/   → crear val y test desde train
#  C) Tiene Training/ Testing/    → renombrar a train/ test/
#  D) Solo carpetas por clase     → split automático 70/15/15
#  E) Tiene __MACOSX u otros junk → ignorar
#  F) PlantVillage color/gray/seg → entrar a color/
#  G) Nivel extra (dataset/)      → bajar un nivel

def es_carpeta_clase(path):
    """True si la carpeta contiene imágenes directamente (es una clase)."""
    imgs = list(path.glob('*.jpg')) + list(path.glob('*.jpeg')) + \
           list(path.glob('*.png')) + list(path.glob('*.JPG'))
    return len(imgs) > 0

def ignorar(nombre):
    """Carpetas/archivos a saltar."""
    return nombre.startswith('__') or nombre.startswith('.') or \
           nombre.lower() in ['grayscale','segmented'] or \
           nombre.lower().endswith('.csv') or nombre.lower().endswith('.txt') or \
           nombre.lower().endswith('.h5') or nombre.lower().endswith('.json')

def encontrar_raiz(base):
    """
    Navega hacia abajo hasta encontrar la raíz útil del dataset.
    Retorna el Path que contiene train/, clase/, etc.
    """
    carpetas = [p for p in base.iterdir()
                if p.is_dir() and not ignorar(p.name)]

    # PlantVillage: tiene color/, grayscale/, segmented/ → entrar a color/
    nombres = {p.name.lower() for p in carpetas}
    if 'color' in nombres:
        print('   Detectado: PlantVillage → usando carpeta color/')
        return encontrar_raiz(base / next(p for p in carpetas if p.name.lower()=='color'))

    # Una sola carpeta contenedora (ej: 'dataset/', 'chest_xray/') → bajar
    if len(carpetas) == 1:
        unica = carpetas[0]
        sub   = [p for p in unica.iterdir() if p.is_dir() and not ignorar(p.name)]
        if sub:  # tiene subcarpetas → bajar
            print(f'   Bajando un nivel: {unica.name}/')
            return encontrar_raiz(unica)

    return base


def detectar_estructura(raiz):
    """
    Devuelve el tipo de estructura:
    'splits'    → ya tiene train/ + val/ (o train/ + test/)
    'clases'    → solo carpetas de clases, sin splits
    """
    carpetas = [p for p in raiz.iterdir()
                if p.is_dir() and not ignorar(p.name)]
    nombres  = {p.name.lower() for p in carpetas}

    splits_conocidos = {'train','training','val','valid','validation',
                        'test','testing'}
    tiene_splits = bool(nombres & splits_conocidos)

    if tiene_splits:
        return 'splits', carpetas
    else:
        # Verificar que las carpetas contienen imágenes (son clases)
        clases = [p for p in carpetas if es_carpeta_clase(p)]
        if clases:
            return 'clases', clases
        # Subcarpetas que a su vez tienen imágenes (ej: Garbage classification/)
        for p in carpetas:
            sub = [s for s in p.iterdir() if s.is_dir() and es_carpeta_clase(s)]
            if sub:
                print(f'   Encontrado contenedor extra: {p.name}/ → usando sus subcarpetas')
                return 'clases', sub
        return 'desconocido', carpetas


def copiar_split(src_clase, dst_split, clase_nombre, porcentaje, seed):
    """Mueve un porcentaje de imágenes de src a dst/clase_nombre/."""
    dst = dst_split / clase_nombre
    dst.mkdir(parents=True, exist_ok=True)
    imgs = (list(src_clase.glob('*.jpg')) + list(src_clase.glob('*.jpeg')) +
            list(src_clase.glob('*.png')) + list(src_clase.glob('*.JPG')))
    random.seed(seed)
    random.shuffle(imgs)
    n = max(1, int(len(imgs) * porcentaje))
    seleccionadas = imgs[:n]
    for img in seleccionadas:
        shutil.move(str(img), str(dst / img.name))
    return n


def normalizar_splits(raiz, train_dir, val_dir, test_dir):
    """
    Recibe carpetas de splits existentes y las renombra/completa
    al formato estándar: train/, val/, test/
    """
    nombres = {p.name.lower(): p for p in raiz.iterdir()
               if p.is_dir() and not ignorar(p.name)}

    # Renombrar Training → train, Testing → test, etc.
    mapa = {'training':'train','testing':'test',
            'validation':'val','valid':'val'}
    for viejo, nuevo in mapa.items():
        if viejo in nombres:
            origen = nombres[viejo]
            destino = raiz / nuevo
            if not destino.exists():
                origen.rename(destino)
                print(f'   Renombrado: {viejo}/ → {nuevo}/')
                nombres[nuevo] = destino

    # Si hay train pero no val → crear val y test desde train
    if 'train' in nombres and 'val' not in nombres:
        print('   Sin val/ → creando val/ y test/ desde train/ (15% cada uno)')
        train_src = nombres['train']
        for clase_dir in sorted(train_src.iterdir()):
            if not clase_dir.is_dir() or ignorar(clase_dir.name): continue
            # val: 15% de train
            n_val  = copiar_split(clase_dir, val_dir,  clase_dir.name, VAL_SIZE,  SEED)
            # test: 15% de train (del resto)
            n_test = copiar_split(clase_dir, test_dir, clase_dir.name, TEST_SIZE, SEED+1)
    elif 'test' in nombres and 'val' not in nombres:
        # tiene test pero no val → crear val desde test
        print('   Sin val/ → creando val/ desde test/ (50% de test)')
        test_src = nombres['test']
        for clase_dir in sorted(test_src.iterdir()):
            if not clase_dir.is_dir() or ignorar(clase_dir.name): continue
            copiar_split(clase_dir, val_dir, clase_dir.name, 0.5, SEED)


# ── PROCESO PRINCIPAL ─────────────────────────────────────────────
print('\n🔍 Analizando estructura del dataset...')
raiz = encontrar_raiz(ZIP_TMP)
print(f'   Raíz detectada: {raiz.relative_to(ZIP_TMP)}')

tipo, carpetas = detectar_estructura(raiz)
print(f'   Tipo de estructura: {tipo}')

TRAIN_DIR = IMAGES_PATH / 'train'
VAL_DIR   = IMAGES_PATH / 'val'
TEST_DIR  = IMAGES_PATH / 'test'

if tipo == 'splits':
    print('\n📁 Estructura con splits detectada — normalizando...')
    # Mover todo a IMAGES_PATH
    for carpeta in carpetas:
        destino = IMAGES_PATH / carpeta.name
        if not destino.exists():
            shutil.copytree(str(carpeta), str(destino))
    normalizar_splits(IMAGES_PATH, TRAIN_DIR, VAL_DIR, TEST_DIR)

elif tipo == 'clases':
    print(f'\n📁 {len(carpetas)} clases detectadas — creando splits 70/15/15...')
    for clase_dir in sorted(carpetas):
        imgs = (list(clase_dir.glob('*.jpg')) + list(clase_dir.glob('*.jpeg')) +
                list(clase_dir.glob('*.png')) + list(clase_dir.glob('*.JPG')))
        if not imgs: continue
        random.seed(SEED); random.shuffle(imgs)
        n_total = len(imgs)
        n_val   = max(1, int(n_total * VAL_SIZE))
        n_test  = max(1, int(n_total * TEST_SIZE))
        n_train = n_total - n_val - n_test
        splits_imgs = {
            'train': imgs[:n_train],
            'val':   imgs[n_train:n_train+n_val],
            'test':  imgs[n_train+n_val:]
        }
        for split_name, split_imgs in splits_imgs.items():
            dst = IMAGES_PATH / split_name / clase_dir.name
            dst.mkdir(parents=True, exist_ok=True)
            for img in split_imgs:
                shutil.copy2(str(img), str(dst / img.name))

else:
    raise ValueError(
        f'No se pudo detectar la estructura del dataset.\n'
        f'Carpetas encontradas: {[p.name for p in carpetas]}\n'
        f'Ejecuta la CELDA 2b para inspeccionar manualmente.'
    )

# Limpiar tmp
shutil.rmtree(str(ZIP_TMP), ignore_errors=True)

# ── Verificación ─────────────────────────────────────────────────
print('\n🔍 Verificación final:')
total_imgs = 0
for split in ['train','val','test']:
    split_path = IMAGES_PATH / split
    if not split_path.exists():
        print(f'   ❌ {split}/ no encontrado'); continue
    clases = [p for p in split_path.iterdir() if p.is_dir()]
    n_imgs = sum(len(list(c.rglob('*.jpg')) + list(c.rglob('*.png')) +
                     list(c.rglob('*.jpeg'))) for c in clases)
    total_imgs += n_imgs
    print(f'   ✅ {split:<6}: {len(clases):>3} clases · {n_imgs:>6,} imágenes')
print(f'\n   Total: {total_imgs:,} imágenes')
print('\n🟢 Dataset listo. Continúa con CELDA 3.')

---
## CELDA 2b — Diagnóstico (ejecuta solo si CELDA 2 falló)

In [ ]:
# ================================================================
# 🔎 DIAGNÓSTICO — solo si CELDA 2 falló
# ================================================================
print('DIAGNÓSTICO DE ESTRUCTURA')
print('='*55)

def mostrar_arbol(path, nivel=0, max_nivel=3, max_items=8):
    if nivel > max_nivel: return
    try:
        items = sorted(path.iterdir())[:max_items]
    except: return
    for item in items:
        prefijo = '  ' * nivel
        if item.is_dir():
            n_imgs = len(list(item.rglob('*.jpg'))+list(item.rglob('*.png')))
            print(f'{prefijo}📂 {item.name}/  ({n_imgs} imgs)')
            mostrar_arbol(item, nivel+1, max_nivel, max_items)
        else:
            if not ignorar(item.name):
                print(f'{prefijo}📄 {item.name}')

print(f'\nContenido de {IMAGES_PATH}:')
mostrar_arbol(IMAGES_PATH)

if ZIP_TMP.exists():
    print(f'\nContenido de {ZIP_TMP} (ZIP sin procesar):')
    mostrar_arbol(ZIP_TMP)

---
## CELDA 3 — Análisis visual del dataset

In [ ]:
# ================================================================
# 📊 DISTRIBUCIÓN DE CLASES
# ================================================================
clases_train = sorted([p.name for p in (IMAGES_PATH/'train').iterdir() if p.is_dir()])
NUM_CLASES   = len(clases_train)
clase2idx    = {c: i for i, c in enumerate(clases_train)}
idx2clase    = {str(i): c for c, i in clase2idx.items()}

# Conteo por split
conteos = {}
for split in ['train','val','test']:
    sp = IMAGES_PATH / split
    if not sp.exists(): continue
    for clase_dir in sp.iterdir():
        if not clase_dir.is_dir(): continue
        n = len(list(clase_dir.rglob('*.jpg')) + list(clase_dir.rglob('*.png')) +
                list(clase_dir.rglob('*.jpeg')))
        conteos.setdefault(clase_dir.name, {})[split] = n

df_dist = pd.DataFrame(conteos).T.fillna(0).astype(int)
df_dist['total'] = df_dist.sum(axis=1)
df_dist = df_dist.sort_values('total', ascending=True)

# Gráfica
fig, axes = plt.subplots(1, 2, figsize=(16, max(5, NUM_CLASES * 0.3)))
fig.suptitle(f'Dataset #{DATASET_ID}: {cfg["nombre"]} — {NUM_CLASES} clases',
             fontsize=13, fontweight='bold')

colores = ['#e74c3c' if v<50 else '#f39c12' if v<200 else '#27ae60'
           for v in df_dist['total'].values]
axes[0].barh(df_dist.index, df_dist['total'], color=colores)
axes[0].set_xlabel('Imágenes totales (train+val+test)')
axes[0].set_title('Distribución de clases')
for i, v in enumerate(df_dist['total']):
    axes[0].text(v+1, i, str(v), va='center', fontsize=8)
patches = [mpatches.Patch(color='#e74c3c',label='<50'),
           mpatches.Patch(color='#f39c12',label='50-200'),
           mpatches.Patch(color='#27ae60',label='>200')]
axes[0].legend(handles=patches, fontsize=8)

# Pie de splits
splits_total = {s: df_dist[s].sum() for s in ['train','val','test'] if s in df_dist.columns}
axes[1].pie(splits_total.values(), labels=splits_total.keys(),
            autopct='%1.1f%%', colors=['#3498db','#e67e22','#2ecc71'],
            startangle=90, textprops={'fontsize':11})
axes[1].set_title('Distribución train / val / test')

plt.tight_layout()
out = BASE_PATH / 'data' / 'distribucion.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'📁 Guardado: {out}')

# Resumen
print(f'\n📊 Resumen:')
print(f'   Clases    : {NUM_CLASES}')
for s, n in splits_total.items():
    print(f'   {s:<8}: {n:,} imágenes')

# Guardar mapeo
with open(BASE_PATH/'data'/'idx2clase.json','w',encoding='utf-8') as f:
    json.dump(idx2clase, f, ensure_ascii=False, indent=2)
print(f'\n✅ idx2clase.json guardado ({NUM_CLASES} clases)')

In [ ]:
# ================================================================
# 🖼️  MUESTRAS DEL DATASET — 3 imágenes por clase (máx 10 clases)
# ================================================================
n_mostrar   = min(10, NUM_CLASES)
clases_vis  = random.sample(clases_train, n_mostrar)
IMGS_X_FILA = 3

fig, axes = plt.subplots(n_mostrar, IMGS_X_FILA,
                         figsize=(IMGS_X_FILA*3, n_mostrar*3))
if n_mostrar == 1: axes = [axes]
fig.suptitle(f'Muestras del dataset — {n_mostrar} clases (máx 10)',
             fontsize=12, fontweight='bold')

for fila, clase in enumerate(clases_vis):
    imgs = (list((IMAGES_PATH/'train'/clase).glob('*.jpg')) +
            list((IMAGES_PATH/'train'/clase).glob('*.png')))
    random.shuffle(imgs)
    for col in range(IMGS_X_FILA):
        ax = axes[fila][col] if n_mostrar > 1 else axes[col]
        if col < len(imgs):
            try:
                ax.imshow(Image.open(imgs[col]).convert('RGB'))
            except: pass
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(clase[:20], fontsize=8, rotation=0,
                          labelpad=60, va='center')

plt.tight_layout()
out = BASE_PATH / 'data' / 'muestras_dataset.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'📁 Guardado: {out}')

---
## CELDA 4 — Augmentación y DataLoaders

In [ ]:
# ================================================================
# 🎨 AUGMENTACIÓN + DATALOADERS
# ================================================================
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_aug = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.4),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.4),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.05, p=0.5),
    A.CLAHE(clip_limit=2.0, p=0.3),
    A.GaussianBlur(blur_limit=(3,5), p=0.2),
    A.GaussNoise(var_limit=(5,20), p=0.2),
    A.CoarseDropout(max_holes=4, max_height=20, max_width=20, p=0.2),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2()
])
val_aug = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2()
])

class FolderDataset(Dataset):
    """Carga imágenes desde carpetas train/val/test/clase/."""
    def __init__(self, split_path, clase2idx, transform=None):
        self.samples   = []
        self.transform = transform
        for clase_dir in sorted(split_path.iterdir()):
            if not clase_dir.is_dir(): continue
            idx = clase2idx.get(clase_dir.name)
            if idx is None: continue
            for ext in ['*.jpg','*.jpeg','*.png','*.JPG','*.JPEG','*.PNG']:
                for img_path in clase_dir.glob(ext):
                    self.samples.append((img_path, idx))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            img = np.array(Image.open(path).convert('RGB'))
        except Exception:
            img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        if self.transform:
            img = self.transform(image=img)['image']
        return img, torch.tensor(label, dtype=torch.long)

train_ds = FolderDataset(IMAGES_PATH/'train', clase2idx, train_aug)
val_ds   = FolderDataset(IMAGES_PATH/'val',   clase2idx, val_aug)
test_ds  = FolderDataset(IMAGES_PATH/'test',  clase2idx, val_aug)

# WeightedRandomSampler — balancea clases desiguales
etiquetas_train = [s[1] for s in train_ds.samples]
conteo_clases   = [etiquetas_train.count(i) for i in range(NUM_CLASES)]
pesos           = [1.0 / max(conteo_clases[lbl], 1) for lbl in etiquetas_train]
sampler         = WeightedRandomSampler(pesos, len(train_ds), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE=='cuda'),
    persistent_workers=(NUM_WORKERS>0))
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE=='cuda'),
    persistent_workers=(NUM_WORKERS>0))
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(DEVICE=='cuda'),
    persistent_workers=(NUM_WORKERS>0))

print('✅ DataLoaders listos')
print(f'   train : {len(train_ds):,} imágenes · {len(train_loader)} batches')
print(f'   val   : {len(val_ds):,} imágenes · {len(val_loader)} batches')
print(f'   test  : {len(test_ds):,} imágenes · {len(test_loader)} batches')
print(f'   WeightedSampler activo — balancea {NUM_CLASES} clases')

---
## CELDA 5 — Modelo

In [ ]:
# ================================================================
# 🧠 MODELO — Transfer Learning
# ================================================================
class ClasificadorCNN(nn.Module):
    def __init__(self, num_classes, model_name=MODEL_NAME, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            model_name, pretrained=pretrained,
            num_classes=0, global_pool='avg'
        )
        in_f = self.backbone.num_features
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(in_f, min(512, in_f)),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(min(512, in_f), num_classes)
        )

    def forward(self, x):
        return self.classifier(self.backbone(x))


model = ClasificadorCNN(NUM_CLASES).to(DEVICE)
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✅ Modelo: {MODEL_NAME}')
print(f'   Parámetros totales     : {total/1e6:.1f} M')
print(f'   Parámetros entrenables : {trainable/1e6:.1f} M')
print(f'   Clases de salida       : {NUM_CLASES}')

with torch.no_grad():
    dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
    out   = model(dummy)
    print(f'   Forward pass OK        : {list(dummy.shape)} → {list(out.shape)}')
del dummy, out
torch.cuda.empty_cache() if DEVICE == 'cuda' else None

---
## CELDA 6 — Entrenamiento

In [ ]:
# ================================================================
# 🚀 ENTRENAMIENTO
# ================================================================
cc = [etiquetas_train.count(i) for i in range(NUM_CLASES)]
cls_weights = torch.tensor(
    [len(etiquetas_train) / (NUM_CLASES * max(c,1)) for c in cc],
    dtype=torch.float32
).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=cls_weights, label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-3)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)

acc_m = Accuracy(task='multiclass', num_classes=NUM_CLASES).to(DEVICE)
f1_m  = F1Score(task='multiclass', num_classes=NUM_CLASES, average='macro').to(DEVICE)

MODEL_SAVE = BASE_PATH / 'models' / 'modelo_best.pth'


def train_epoch():
    model.train(); total_loss = 0.0; acc_m.reset()
    for imgs, labels in train_loader:
        imgs   = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        acc_m.update(logits.detach(), labels)
    return total_loss / len(train_loader), acc_m.compute().item()


@torch.no_grad()
def evaluate(loader):
    model.eval(); total_loss = 0.0; acc_m.reset(); f1_m.reset()
    for imgs, labels in loader:
        imgs   = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        logits = model(imgs)
        total_loss += criterion(logits, labels).item()
        acc_m.update(logits, labels)
        f1_m.update(logits, labels)
    return total_loss/len(loader), acc_m.compute().item(), f1_m.compute().item()


best_acc, no_imp, history = 0.0, 0, []
sep = '─' * 78
print(f'🚀 Entrenando: {EPOCHS} epochs | patience={PATIENCE} | device={DEVICE}')
print(f'   Dataset : #{DATASET_ID} {cfg["nombre"]}')
print(f'   Modelo  : {MODEL_NAME} | {NUM_CLASES} clases')
print(sep)
print(f'  {"Epoch":>5} | {"Train Loss":>10} | {"Train Acc":>9} | {"Val Loss":>8} | {"Val Acc":>7} | {"Val F1":>6}')
print(sep)

for epoch in range(1, EPOCHS+1):
    tl, ta      = train_epoch()
    vl, va, vf1 = evaluate(val_loader)
    scheduler.step(epoch)
    history.append({'epoch':epoch,'train_loss':tl,'train_acc':ta,
                    'val_loss':vl,'val_acc':va,'val_f1':vf1})
    mejoro = va > best_acc
    print(f'  {epoch:5d} | {tl:10.4f} | {ta:9.4f} | {vl:8.4f} | {va:7.4f} | {vf1:6.4f} {"🟢" if mejoro else ""}')
    if mejoro:
        best_acc = va; no_imp = 0
        torch.save({
            'epoch':epoch,'model_state_dict':model.state_dict(),
            'val_acc':va,'val_f1':vf1,'idx2clase':idx2clase,
            'num_classes':NUM_CLASES,'model_name':MODEL_NAME,
            'img_size':IMG_SIZE,'mean':MEAN,'std':STD,
            'dataset_id':DATASET_ID,'dataset_nombre':cfg['nombre'],
        }, MODEL_SAVE)
        print(f'         💾 Checkpoint guardado — acc={va:.4f} f1={vf1:.4f}')
    else:
        no_imp += 1
        if no_imp >= PATIENCE:
            print(f'\n⛔ Early stopping en epoch {epoch}')
            break

pd.DataFrame(history).to_csv(BASE_PATH/'data'/'historial.csv', index=False)
print(f'\n🏆 Entrenamiento finalizado')
print(f'   Mejor val_acc : {best_acc:.4f} ({best_acc*100:.2f}%)')

---
## CELDA 7 — Gráficas de entrenamiento

In [ ]:
# ================================================================
# 📈 CURVAS DE ENTRENAMIENTO
# ================================================================
hist_df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle(f'#{DATASET_ID} {cfg["nombre"]} · {MODEL_NAME} · {NUM_CLASES} clases',
             fontsize=12, y=1.02)

best_ep = hist_df.loc[hist_df['val_acc'].idxmax(), 'epoch']

for ax, y1, y2, title in [
    (axes[0], 'train_loss', 'val_loss', 'Loss'),
    (axes[1], 'train_acc',  'val_acc',  'Accuracy'),
]:
    ax.plot(hist_df['epoch'], hist_df[y1], label='Train', color='#3498db', lw=2)
    ax.plot(hist_df['epoch'], hist_df[y2], label='Val',   color='#e74c3c', lw=2, ls='--')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Época'); ax.legend(); ax.grid(alpha=0.3)

axes[1].axvline(x=best_ep, color='#27ae60', ls='--', alpha=0.8,
                label=f'Mejor epoch ({int(best_ep)})')
axes[1].legend()

axes[2].plot(hist_df['epoch'], hist_df['val_f1'], color='#9b59b6', lw=2)
axes[2].fill_between(hist_df['epoch'], hist_df['val_f1'], alpha=0.15, color='#9b59b6')
axes[2].set_title('F1 Macro (Val)', fontweight='bold')
axes[2].set_xlabel('Época'); axes[2].grid(alpha=0.3)

plt.tight_layout()
out = BASE_PATH / 'data' / 'curvas.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'📁 Guardado: {out}')

---
## CELDA 8 — Evaluación en Test Set

In [ ]:
# ================================================================
# 🔬 EVALUACIÓN — Test Set
# ================================================================
ckpt = torch.load(MODEL_SAVE, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'✅ Mejor modelo cargado (epoch {ckpt["epoch"]}, val_acc={ckpt["val_acc"]:.4f})')

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        logits = model(imgs.to(DEVICE))
        probs  = F.softmax(logits, dim=1).cpu().numpy()
        preds  = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
        all_probs.extend(probs)

nombres_clases = [idx2clase[str(i)] for i in range(NUM_CLASES)]
test_acc = (np.array(all_preds) == np.array(all_labels)).mean()
print(f'\n🎯 Test Accuracy : {test_acc:.4f} ({test_acc*100:.2f}%)')
print('\n📊 Reporte de clasificación:')
reporte = classification_report(all_labels, all_preds,
                                 target_names=nombres_clases, digits=3, output_dict=True)
print(classification_report(all_labels, all_preds,
                              target_names=nombres_clases, digits=3))
# Guardar reporte CSV
df_rep = pd.DataFrame(reporte).T
df_rep.to_csv(BASE_PATH/'data'/'reporte.csv')
print(f'📁 reporte.csv guardado')

In [ ]:
# ================================================================
# 🟦 MATRICES DE CONFUSIÓN
# ================================================================
cm      = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

anotar  = NUM_CLASES <= 15
fig_w   = max(14, NUM_CLASES * 0.6)
fig, axes = plt.subplots(1, 2, figsize=(fig_w*2, fig_w))

for ax, data, fmt, cmap, title in [
    (axes[0], cm,      'd',   'Blues',  'Conteos absolutos'),
    (axes[1], cm_norm, '.2f', 'YlOrRd', 'Normalizada (Recall por clase)'),
]:
    sns.heatmap(data, ax=ax, cmap=cmap, annot=anotar, fmt=fmt,
                xticklabels=nombres_clases, yticklabels=nombres_clases)
    ax.set_title(f'Matriz de Confusión — {title}', fontweight='bold')
    ax.set_xlabel('Predicho'); ax.set_ylabel('Real')
    ax.tick_params(axis='x', rotation=45, labelsize=max(5, 9-NUM_CLASES//10))
    ax.tick_params(axis='y', rotation=0,  labelsize=max(5, 9-NUM_CLASES//10))

plt.tight_layout()
for fname, ax_idx in [('confusion_counts.png',0),('confusion_norm.png',1)]:
    fig2, ax2 = plt.subplots(figsize=(fig_w, fig_w))
    data  = cm      if ax_idx == 0 else cm_norm
    fmt   = 'd'     if ax_idx == 0 else '.2f'
    cmap  = 'Blues' if ax_idx == 0 else 'YlOrRd'
    title = 'Conteos' if ax_idx == 0 else 'Normalizada'
    sns.heatmap(data, ax=ax2, cmap=cmap, annot=anotar, fmt=fmt,
                xticklabels=nombres_clases, yticklabels=nombres_clases)
    ax2.set_title(f'Matriz de Confusión — {title}', fontweight='bold')
    ax2.set_xlabel('Predicho'); ax2.set_ylabel('Real')
    plt.tight_layout()
    plt.savefig(BASE_PATH/'data'/fname, dpi=150, bbox_inches='tight')
    plt.close(fig2)

plt.show()
print('📁 confusion_counts.png y confusion_norm.png guardados')

In [ ]:
# ================================================================
# 📊 PRECISION / RECALL / F1 POR CLASE
# ================================================================
clases_plot = nombres_clases
precision_v = [reporte[c]['precision'] for c in clases_plot if c in reporte]
recall_v    = [reporte[c]['recall']    for c in clases_plot if c in reporte]
f1_v        = [reporte[c]['f1-score']  for c in clases_plot if c in reporte]
clases_plot = [c for c in clases_plot if c in reporte]

x = np.arange(len(clases_plot))
w = 0.25
fig, ax = plt.subplots(figsize=(max(12, len(clases_plot)*0.8), 5))
ax.bar(x-w,   precision_v, w, label='Precision', color='#3498db', alpha=0.85)
ax.bar(x,     recall_v,    w, label='Recall',    color='#27ae60', alpha=0.85)
ax.bar(x+w,   f1_v,        w, label='F1-Score',  color='#9b59b6', alpha=0.85)
ax.axhline(y=0.5, color='red',    ls='--', alpha=0.4, lw=1)
ax.axhline(y=0.8, color='green',  ls='--', alpha=0.4, lw=1)
ax.set_xticks(x)
ax.set_xticklabels(clases_plot, rotation=45, ha='right',
                   fontsize=max(6, 10-len(clases_plot)//10))
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title(f'Métricas por clase — #{DATASET_ID} {cfg["nombre"]}', fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
out = BASE_PATH / 'data' / 'metricas_clase.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'📁 Guardado: {out}')

---
## CELDA 9 — Test visual de predicciones

In [ ]:
# ================================================================
# 🖼️  GRID DE 16 PREDICCIONES ALEATORIAS DEL TEST SET
# ================================================================
model.eval()
idxs_muestra = random.sample(range(len(test_ds)), min(16, len(test_ds)))

fig, axes = plt.subplots(2, 8, figsize=(20, 6))
fig.suptitle('Predicciones del test set — verde=correcto · rojo=error',
             fontsize=12, fontweight='bold')

for ax, idx in zip(axes.ravel(), idxs_muestra):
    path, lbl_real = test_ds.samples[idx]
    try:
        img_np = np.array(Image.open(path).convert('RGB'))
        tensor = val_aug(image=img_np)['image'].unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            probs   = F.softmax(model(tensor), dim=1)[0]
            pred    = probs.argmax().item()
            confianza = probs[pred].item() * 100
        ax.imshow(img_np)
        ok    = pred == lbl_real
        color = '#27ae60' if ok else '#e74c3c'
        real  = idx2clase[str(lbl_real)][:12]
        predd = idx2clase[str(pred)][:12]
        ax.set_title(f'{"✅" if ok else "❌"}\n{predd}\n{confianza:.0f}%',
                     fontsize=7, color=color)
    except Exception:
        pass
    ax.axis('off')

plt.tight_layout()
out = BASE_PATH / 'data' / 'predicciones_grid.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'📁 Guardado: {out}')

In [ ]:
# ================================================================
# 🖼️  UNA MUESTRA POR CLASE CON TOP-3
# ================================================================
n_vis   = min(NUM_CLASES, 12)
clases_v = clases_train[:n_vis]

fig, axes = plt.subplots(2, (n_vis+1)//2, figsize=(20, 9))
fig.suptitle('Una muestra por clase — top-3 predicciones',
             fontsize=12, fontweight='bold')

for ax, clase in zip(axes.ravel(), clases_v):
    muestras = [(p, l) for p, l in test_ds.samples if l == clase2idx[clase]]
    if not muestras:
        ax.axis('off'); continue
    path, lbl_real = random.choice(muestras)
    try:
        img_np = np.array(Image.open(path).convert('RGB'))
        tensor = val_aug(image=img_np)['image'].unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            probs        = F.softmax(model(tensor), dim=1)[0]
            top3p, top3i = torch.topk(probs, min(3, NUM_CLASES))
        ax.imshow(img_np)
        ax.axis('off')
        pred_ok = top3i[0].item() == lbl_real
        txt = '\n'.join([
            f"{idx2clase[str(i.item())][:14]}: {p*100:.0f}%"
            for p, i in zip(top3p, top3i)
        ])
        ax.set_title(f'Real: {clase[:14]}\n{"✅" if pred_ok else "❌"}\n{txt}',
                     fontsize=6.5,
                     color='#27ae60' if pred_ok else '#e74c3c')
    except Exception:
        ax.axis('off')

for ax in axes.ravel()[n_vis:]:
    ax.axis('off')

plt.tight_layout()
out = BASE_PATH / 'data' / 'muestras_por_clase.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'📁 Guardado: {out}')

---
## CELDA 10A — Probar con una imagen del test set (al azar)


In [ ]:
# ================================================================
# 🎲 OPCIÓN A — Imagen aleatoria del test set
# ================================================================
model.eval()
path_test, lbl_real = random.choice(test_ds.samples)

img_np = np.array(Image.open(path_test).convert('RGB'))
tensor = val_aug(image=img_np)['image'].unsqueeze(0).to(DEVICE)

with torch.no_grad():
    probs        = F.softmax(model(tensor), dim=1)[0].cpu()
    top5p, top5i = torch.topk(probs, min(5, NUM_CLASES))

nombre_real = idx2clase[str(lbl_real)]
nombre_pred = idx2clase[str(top5i[0].item())]
correcto    = nombre_pred == nombre_real

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(f'Test con imagen del dataset — {"✅ CORRECTO" if correcto else "❌ INCORRECTO"}',
             fontsize=12, fontweight='bold',
             color='#27ae60' if correcto else '#e74c3c')

axes[0].imshow(img_np)
axes[0].set_title(f'Real: {nombre_real}', fontsize=11)
axes[0].axis('off')

top_nombres = [idx2clase[str(i.item())][:20] for i in top5i]
top_probs   = [p.item()*100 for p in top5p]
colores_bar = ['#27ae60' if n==nombre_real else '#3498db'
               for n in top_nombres]
bars = axes[1].barh(top_nombres[::-1], top_probs[::-1],
                    color=colores_bar[::-1], edgecolor='white')
for bar, prob in zip(bars, top_probs[::-1]):
    axes[1].text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2,
                 f'{prob:.1f}%', va='center', fontsize=10)
axes[1].set_xlim(0, 110)
axes[1].set_xlabel('Probabilidad (%)')
axes[1].set_title('Top-5 predicciones', fontsize=11)
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(BASE_PATH/'data'/'test_aleatorio.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Real      : {nombre_real}')
print(f'Predicción: {nombre_pred} ({top5p[0].item()*100:.1f}%)')
print(f'Resultado : {"✅ CORRECTO" if correcto else "❌ INCORRECTO"}')

---
## CELDA 10B — Probar con tu propia imagen
> Sube cualquier imagen desde tu PC — el modelo intentará clasificarla.

In [ ]:
# ================================================================
# 📸 OPCIÓN B — Sube tu propia imagen y clasifícala
# ================================================================
model.eval()

if ENTORNO == 'colab':
    print('📂 Selecciona una imagen desde tu PC...')
    from google.colab import files
    subida  = files.upload()
    img_path = Path(list(subida.keys())[0])
else:
    # En local: cambia esta ruta a tu imagen
    img_path = Path(r'C:\ruta\a\tu\imagen.jpg')
    if not img_path.exists():
        raise FileNotFoundError(
            f'Imagen no encontrada: {img_path}\n'
            'Cambia img_path a la ruta de tu imagen.'
        )

img_np = np.array(Image.open(img_path).convert('RGB'))
tensor = val_aug(image=img_np)['image'].unsqueeze(0).to(DEVICE)

with torch.no_grad():
    probs        = F.softmax(model(tensor), dim=1)[0].cpu()
    top5p, top5i = torch.topk(probs, min(5, NUM_CLASES))

nombre_pred = idx2clase[str(top5i[0].item())]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(f'Tu imagen — Clasificada como: {nombre_pred}',
             fontsize=12, fontweight='bold')

axes[0].imshow(img_np)
axes[0].set_title(img_path.name, fontsize=10)
axes[0].axis('off')

top_nombres = [idx2clase[str(i.item())][:20] for i in top5i]
top_probs   = [p.item()*100 for p in top5p]
axes[1].barh(top_nombres[::-1], top_probs[::-1],
             color='#0C447C', alpha=0.85, edgecolor='white')
for i, prob in enumerate(top_probs[::-1]):
    axes[1].text(prob+0.5, i, f'{prob:.1f}%', va='center', fontsize=10)
axes[1].set_xlim(0, 110)
axes[1].set_xlabel('Probabilidad (%)')
axes[1].set_title('Top-5 predicciones', fontsize=11)
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
out = BASE_PATH / 'data' / 'mi_prediccion.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()

print(f'\n🎯 Resultado:')
for i, (p, idx) in enumerate(zip(top5p, top5i)):
    print(f'   Top-{i+1}: {idx2clase[str(idx.item())]:<25} {p.item()*100:.2f}%')
print(f'\n📁 Guardado: {out}')

---
## CELDA 11 — Exportar modelo para producción

In [ ]:
# ================================================================
# 📦 EXPORTAR MODELO
# ================================================================
ckpt = torch.load(MODEL_SAVE, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

# 1. PyTorch con todos los metadatos
prod_path = BASE_PATH / 'models' / 'modelo_produccion.pth'
torch.save({
    'model_state_dict': model.state_dict(),
    'idx2clase':    idx2clase,
    'num_classes':  NUM_CLASES,
    'model_name':   MODEL_NAME,
    'img_size':     IMG_SIZE,
    'mean':         MEAN,
    'std':          STD,
    'val_acc':      ckpt['val_acc'],
    'val_f1':       ckpt['val_f1'],
    'best_epoch':   ckpt['epoch'],
    'dataset_id':   DATASET_ID,
    'dataset_nombre': cfg['nombre'],
}, prod_path)
print(f'✅ PyTorch  : {prod_path}')

# 2. ONNX
onnx_path = BASE_PATH / 'models' / 'modelo.onnx'
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
with torch.no_grad():
    torch.onnx.export(
        model, dummy, str(onnx_path),
        export_params=True, opset_version=17,
        do_constant_folding=True,
        input_names=['image'], output_names=['logits'],
        dynamic_axes={'image':{0:'batch'},'logits':{0:'batch'}}
    )
del dummy
print(f'✅ ONNX     : {onnx_path}')

# 3. idx2clase.json
with open(BASE_PATH/'data'/'idx2clase.json','w',encoding='utf-8') as f:
    json.dump(idx2clase, f, ensure_ascii=False, indent=2)
print(f'✅ idx2clase.json')

print('\n📋 Tamaños:')
for r in [prod_path, onnx_path]:
    print(f'   {r.name:35s} {r.stat().st_size/1e6:6.1f} MB')

---
## CELDA 12 — Guardar resultados
> **Colab:** guarda automáticamente en Drive. **Local:** los archivos ya están en `BASE_PATH`.

In [ ]:
# ================================================================
# ☁️  GUARDAR RESULTADOS
# ================================================================
archivos_guardar = [
    BASE_PATH/'models'/'modelo_produccion.pth',
    BASE_PATH/'models'/'modelo.onnx',
    BASE_PATH/'data'/'idx2clase.json',
    BASE_PATH/'data'/'historial.csv',
    BASE_PATH/'data'/'reporte.csv',
    BASE_PATH/'data'/'curvas.png',
    BASE_PATH/'data'/'distribucion.png',
    BASE_PATH/'data'/'muestras_dataset.png',
    BASE_PATH/'data'/'confusion_counts.png',
    BASE_PATH/'data'/'confusion_norm.png',
    BASE_PATH/'data'/'metricas_clase.png',
    BASE_PATH/'data'/'predicciones_grid.png',
    BASE_PATH/'data'/'muestras_por_clase.png',
]

if ENTORNO == 'colab':
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    drive_out = Path(f'/content/drive/{DRIVE_OUTPUT}')
    drive_out.mkdir(parents=True, exist_ok=True)
    print(f'Guardando en Drive: {drive_out}')
    for ruta in archivos_guardar:
        if ruta.exists():
            shutil.copy(str(ruta), str(drive_out / ruta.name))
            print(f'  ✅ {ruta.name:<40} {ruta.stat().st_size/1e6:5.1f} MB')
        else:
            print(f'  ⚠️  {ruta.name} — no encontrado')
    print(f'\n✅ Todo guardado en Drive')
else:
    print(f'Entorno local — archivos ya en: {BASE_PATH}')
    print('\nArchivos generados:')
    for ruta in archivos_guardar:
        existe = ruta.exists()
        size   = f'{ruta.stat().st_size/1e6:.1f} MB' if existe else 'no encontrado'
        print(f'  {"✅" if existe else "⚠️"} {ruta.name:<40} {size}')

---
## CELDA 13 — Resumen final

In [ ]:
# ================================================================
# 📋 RESUMEN FINAL
# ================================================================
ckpt = torch.load(MODEL_SAVE, map_location='cpu')
print('=' * 60)
print(f'  🤖 ACTIVIDAD FINAL — CLASIFICACIÓN CNN')
print('=' * 60)
print(f'  Dataset       : #{DATASET_ID} {cfg["nombre"]}')
print(f'  Modelo        : {MODEL_NAME}')
print(f'  Clases        : {NUM_CLASES}')
print(f'  Imágenes      : train={len(train_ds):,} | val={len(val_ds):,} | test={len(test_ds):,}')
print(f'  Mejor epoch   : {ckpt["epoch"]}')
print(f'  Val Accuracy  : {ckpt["val_acc"]*100:.2f}%')
print(f'  Val F1 Macro  : {ckpt["val_f1"]*100:.2f}%')
print(f'  Test Accuracy : {test_acc*100:.2f}%')
print('=' * 60)
print('  Archivos generados:')
for ruta in archivos_guardar:
    if ruta.exists():
        print(f'  ✅ {ruta.name}')
print('=' * 60)
print('  Próximos pasos:')
print('  1. Sube modelo_produccion.pth a tu repo de GitHub')
print('  2. Despliega la FastAPI en Render')
print('  3. Conéctala a la app Expo')
print('=' * 60)